In [ ]:

# 1. FUNGSI WRAPPER ADF TEST (Agar Rapi & Reusable)
def check_stationarity(series, name):
    print(f"--- ADF Test Result for {name} ---")
    result = adfuller(series)
    print(f'ADF Statistic: {result[0]:.4f}')
    print(f'p-value: {result[1]:.4f}')
    print('Critical Values:')
    for key, value in result[4].items():
        print(f'   {key}: {value:.4f}')
    
    if result[1] <= 0.05:
        print("Kesimpulan: Data STASIONER (Siap untuk ARIMA)")
    else:
        print("Kesimpulan: Data TIDAK STASIONER (Perlu Differencing)")
    print("-" * 35)

# 2. EKSEKUSI ADF TEST PERTAMA
# Filter baris awal yang NaN akibat shift/rolling (jika ada)
series_original = df_monthly['Claim_Frequency']
check_stationarity(series_original, "Original Frequency")

# 3. FIRST DIFFERENCING (Jika p-value > 0.05)
# Menghitung selisih antar bulan (t - (t-1))
df_monthly['Freq_Diff_1'] = df_monthly['Claim_Frequency'].diff()

# Uji ADF lagi pada data yang sudah di-difference (hapus baris pertama yang NaN)
series_diff = df_monthly['Freq_Diff_1'].dropna()
check_stationarity(series_diff, "First Difference Frequency")

# 4. VISUAL PROOF (Juri sangat suka ini!)
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(df_monthly['Periode_Klaim'], df_monthly['Claim_Frequency'], marker='o', color='blue')
plt.title('Original: Ada Tren/Seasonal?')
plt.xticks(rotation=45)

plt.subplot(1, 2, 2)
plt.plot(df_monthly['Periode_Klaim'], df_monthly['Freq_Diff_1'], marker='o', color='red')
plt.axhline(0, color='black', linestyle='--')
plt.title('Differenced: Terlihat Stationer (Mean 0)')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Ditambah 1 (log1p) untuk antisipasi jika ada angka 0 klaim
df_monthly['Log_Frequency'] = np.log1p(df_monthly['Claim_Frequency'])
df_monthly['Log_Freq_Diff_1'] = df_monthly['Log_Frequency'].diff()

# 2. ADF Test pada Log-Difference
series_to_test = df_monthly['Log_Freq_Diff_1'].dropna()
res_log = adfuller(series_to_test)

print(f"--- HASIL UNTUK MENTOR ---")
print(f"Total Data: {len(df_monthly)} Bulan")
print(f"ADF Statistic (Log-Diff): {res_log[0]:.4f}")
print(f"p-value (Log-Diff): {res_log[1]:.4f}")

# 3. Plot ACF & PACF (Penentuan p, d, q)
fig, ax = plt.subplots(1, 2, figsize=(15, 5))

# ACF untuk menentukan q (Moving Average)
plot_acf(series_to_test, lags=8, ax=ax[0])
ax[0].set_title('ACF - Identifikasi q (MA)')

# PACF untuk menentukan p (Auto-Regressive)
plot_pacf(series_to_test, lags=8, ax=ax[1])
ax[1].set_title('PACF - Identifikasi p (AR)')

plt.tight_layout()
plt.show()